In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Silver Layer: Product Catalog Dimension (`silver.dim_products_catalog`)
# MAGIC 
# MAGIC - **Sources**: `bronze.catalog_data`, `bronze.catalogs_3rd_party`
# MAGIC - **Target**: `silver.dim_products_catalog` (Managed Delta table in Unity Catalog)
# MAGIC - **Key Mechanics**:
# MAGIC   - Dynamically resolves product identifiers (`variant_id`, `item_id`, `product_id`, etc.).
# MAGIC   - Cleans and normalizes brand names, variant names, categories, and pricing.
# MAGIC   - Generates deterministic surrogate key `product_sk` via SHA-256.

# COMMAND ----------

from pyspark.sql import functions as F

# Ensure silver schema exists
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

# COMMAND ----------

# 1. Read available Bronze catalog tables
df_cat_internal = spark.table("bronze.catalog_data") if spark.catalog.tableExists("bronze.catalog_data") else None
df_cat_3rdparty = spark.table("bronze.catalogs_3rd_party") if spark.catalog.tableExists("bronze.catalogs_3rd_party") else None

# COMMAND ----------

def standardize_catalog(df, source_label):
    """Dynamically standardizes catalog schema regardless of slight column variations."""
    cols = df.columns
    
    # Dynamic column resolution
    id_candidates = ["variant_id", "product_id", "prod_id", "item_id", "sku", "variant_name"]
    name_candidates = ["variant_name", "product_name", "prod_name", "item_name", "title"]
    brand_candidates = ["brand_name", "brand", "manufacturer"]
    cat_candidates = ["category", "category_name", "attribute_key", "department"]
    price_candidates = ["price", "unit_price", "mrp", "cost", "selling_price"]
    
    id_col = next((c for c in id_candidates if c in cols), cols[0])
    name_col = next((c for c in name_candidates if c in cols), id_col)
    brand_col = next((c for c in brand_candidates if c in cols), None)
    cat_col = next((c for c in cat_candidates if c in cols), None)
    price_col = next((c for c in price_candidates if c in cols), None)
    
    df_std = (
        df
        .withColumn("product_id", F.trim(F.col(id_col).cast("string")))
        .withColumn("product_name", F.trim(F.col(name_col).cast("string")))
        .withColumn("brand_name", F.trim(F.col(brand_col).cast("string")) if brand_col else F.lit("Unknown"))
        .withColumn("category", F.trim(F.col(cat_col).cast("string")) if cat_col else F.lit("General"))
        .withColumn("unit_price", F.col(price_col).cast("double") if price_col else F.lit(0.0))
        .withColumn("source_catalog", F.lit(source_label))
        .select(
            "product_id",
            "product_name",
            "brand_name",
            "category",
            "unit_price",
            "source_catalog"
        )
        .filter(F.col("product_id").isNotNull() & (F.col("product_id") != ""))
    )
    
    return df_std

# COMMAND ----------

# 2. Standardize available sources
frames_to_union = []

if df_cat_internal is not None:
    frames_to_union.append(standardize_catalog(df_cat_internal, "INTERNAL"))

if df_cat_3rdparty is not None:
    frames_to_union.append(standardize_catalog(df_cat_3rdparty, "3RD_PARTY"))

# Union all standard sources
df_unioned = frames_to_union[0]
for next_df in frames_to_union[1:]:
    df_unioned = df_unioned.unionByName(next_df)

# COMMAND ----------

# 3. Deduplicate and generate Surrogate Key (SHA-256)
df_dim_products = (
    df_unioned
    .dropDuplicates(["product_id"])
    .withColumn(
        "product_sk",
        F.sha2(F.col("product_id"), 256)
    )
    .withColumn("_transformed_at", F.current_timestamp())
    .select(
        "product_sk",
        "product_id",
        "product_name",
        "brand_name",
        "category",
        "unit_price",
        "source_catalog",
        "_transformed_at"
    )
)

# COMMAND ----------

# 4. Save as managed Delta table in Unity Catalog
(
    df_dim_products.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("silver.dim_products_catalog")
)

# Optimize for query speed
spark.sql("OPTIMIZE silver.dim_products_catalog ZORDER BY (category, brand_name)")

print("SUCCESS: silver.dim_products_catalog registered successfully.")

# COMMAND ----------

# 5. Validate output
display(
    spark.table("silver.dim_products_catalog")
    .orderBy("category", "brand_name")
    .limit(10)
)